In [ ]:
import pandas as pd

file_path="/content/drive/MyDrive/Social_Network_Ads.csv"
df= pd.read_csv(file_path)
df.head()

In [ ]:
#check shape
df.shape

In [ ]:
#check data types and null counts
df.info()


In [ ]:
# Check data types

print("\nData Types:")
print(df.dtypes)

OBSERVATION:

No anomoly seen in data types.

In [ ]:
# Display the number of unique values for each column
unique_counts = df.nunique()
print(unique_counts) #all score columns are continuous


OBSERVATION:

Based on the number of unique values, numerical columns can be categorized as following:

continuous: Age, EstimatedSalary
discete   : Purchased

categorcal: Gender

In [ ]:
# Show unique values for each column
categorical_cols = df.select_dtypes(exclude=['number'])
print(categorical_cols)

In [ ]:
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].unique())
    print("-" * 40)   # all good


OBSERVATION:

No odd values fund in categorical columns that need correction

In [ ]:
# duplicate in data set
df.duplicated().sum()

In [ ]:
#overview of missing values

df.isnull().sum()


OBSERVATION:

No missing values found.

In [ ]:
#outliers

import matplotlib.pyplot as plt

continuous_cols = ['Age','EstimatedSalary']

# Plot boxplots in a loop
for col in continuous_cols:
    plt.figure(figsize=(6, 4))
    plt.boxplot(df[col].dropna())
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

OBSERVATION:

No outliers found.

In [ ]:
# encode categorical features

categorical_cols = df.select_dtypes(exclude=['number'])
print(categorical_cols)

In [ ]:
# nominal : gender                         -> one-hot encoding
            #purchased                     -> already binary



OBSERVATION:

Sincebgender is nominal, needs one-hot encodng.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Scatter plot: Age vs Estimated Salary colored by Purchased
sns.scatterplot(data=df, x='Age', y='EstimatedSalary', hue='Purchased', palette='Set1')
plt.title('Age vs Estimated Salary by Purchase')
plt.show()


OBSERVATION:
Out of the first 10 samples:
Only 1 person purchased the product.
That person is:
Age: 32, Salary: 150,000
The rest of the users (with ages ranging from 19 to 35, and salaries from 19k to 84k) did not purchase the product.

CONCLUSION:
Higher salaries may correlate with purchasing behavior:
The only buyer had the highest salary (150,000) in this subset.
Users with lower salaries (<85,000) didn?t purchase, even at similar ages.
Age alone may not be a strong predictor:
People aged 27 (three times) didn?t purchase despite varying salaries.
A younger person with high salary might still purchase (not confirmed from this small sample).
Income appears more influential than age (at least initially).


MODEL PRE OBSERVATIONS:

Classifiers should be able to learn a pattern where high salary increases purchase probability.
Decision Trees may split first on salary thresholds.
KNN might classify high-income points closer to the "Purchased" class.

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='Blues')
plt.title("Correlation Matrix - Social Network Ads")
plt.show()


OBSERVATION:

Age (0.62) ? Strong Positive Correlation
Older users are significantly more likely to purchase.
Age is the most influential predictor in this dataset.

Estimated Salary (0.36) ? Moderate Positive Correlation
Users with higher salaries tend to purchase more, but the effect is less pronounced than Age.

Gender (-0.04) ? Negligible Correlation
Gender has virtually no impact on purchase behavior.
Also shows little correlation with Age and Salary, indicating independence.

CONCLUSION:

Primary Predictors: Age and Estimated Salary are the main drivers of purchase decisions.

Model Implication: Focus on these numeric features; models like KNN and Decision Trees are likely to capture their effects well.

Scatter Plot Insight: Likely shows higher purchase density at higher Age and Salary values, with some overlap near decision boundaries.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Drop User ID
df.drop('User ID', axis=1, inplace=True)

# Encode Gender
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# Features and Target
X = df[['Gender', 'Age', 'EstimatedSalary']]
y = df['Purchased']

# Standardize Age and Salary
sc = StandardScaler()
X[['Age', 'EstimatedSalary']] = sc.fit_transform(X[['Age', 'EstimatedSalary']])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# 1. Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
evaluate_model(nb_model, X_test, y_test, "Naive Bayes")

# 2. K-Nearest Neighbors (k=3,5,7)
for k in [3, 5, 7]:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    evaluate_model(knn_model, X_test, y_test, f"KNN (k={k})")

# 3. Decision Tree (Gini)
dt_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
dt_gini.fit(X_train, y_train)
evaluate_model(dt_gini, X_test, y_test, "Decision Tree (Gini)")

# 4. Decision Tree (Entropy)
dt_entropy = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_entropy.fit(X_train, y_train)
evaluate_model(dt_entropy, X_test, y_test, "Decision Tree (Entropy)")


In [ ]:
def plot_conf_matrix(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {title}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Example for Naive Bayes
plot_conf_matrix(nb_model, X_test, y_test, "Naive Bayes")


Model Performance Summary

Best Model: KNN (k=7)

Accuracy: 0.93 | Precision: 0.875 | Recall: 0.946 | F1-score: 0.909
Strong balance between precision and recall.
Confusion Matrix: 2 false negatives, 5 false positives ? excellent performance.

Naive Bayes:-

Accuracy: 0.93 | Precision: 0.94 | Recall: 0.865 | F1-score: 0.901
Slightly better at minimizing false positives but misses more actual purchasers.
Good alternative for simplicity and speed.


KNN (k=3 and k=5):

Lower accuracy and F1-score than k=7.
Performance improves as k increases, showing better generalization with k=7.


Decision Tree (Gini & Entropy):

Accuracy: ~0.84 | Precision: 0.80 | Recall: 0.757 | F1-score: 0.778
More false predictions (9 false negatives, 7 false positives).
Likely overfitting or less effective with continuous features.


Conclusion:

KNN (k=7) is the most effective model, balancing precision and recall with high accuracy.
Naive Bayes is a strong and efficient alternative.
Decision Trees underperform and may require tuning or additional data.
Models handling continuous features (Age, Salary) like KNN and Naive Bayes outperform Decision Trees.

